In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════
from the_well.data.normalization import ZScoreNormalization
import sys, os, datetime, fsspec, torch, munch, toml, yaml
from the_well.data import WellDataModule
from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
import shutil

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.configs import FlowMatchingConfig, LatentVAEConfig, OptimizerConfig, VelocityNetConfig
from waveletflux.utils.augmentations import compose_augmentation
from waveletflux.utils.well_data import WellDataModuleLightning
from waveletflux.modules.latent_fm import LatentFlowModule

#from waveletflux.utils.configs import WaveletConfig, VelocityNetConfig, FlowMatchingConfig, OptimizerConfig
#from waveletflux.modules.datamodules import WellDataModuleLightning
#from waveletflux.modules.augmentations import compose_augmentation
#from waveletflux.utils.experiment import ExperimentManager
#from waveletflux.wavelet_fm import WaveletFlowModule
#from waveletflux.models.wavelet_unet import WaveletFlowUNet

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

Imports OK | MPS: True
MPS allocated : 1.01 GB
MPS reserved  : 8.09 GB


In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Config
# ═══════════════════════════════════════════════════════════════
CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/LatentFlow_turbulent_radiative_layer_2D_1.0.toml"
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets"
#well_dataset_name = 'rayleigh_benard'
well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"
experiments_path  = f"{PROJECT_ROOT}/experiments/{well_dataset_name}"

config = munch.munchify(toml.load(CONFIG_PATH))

# Override for local MPS run
config.lightning.trainer.accelerator      = "mps"
config.lightning.trainer.strategy         = "auto"
config.lightning.trainer.precision        = "32"
config.lightning.trainer.matmul_precision = "high"
#config.train.batch_size                   = 1  # default from the config file
#config.train.epochs                       = 2  # default from the config file

pl.seed_everything(config.random.seed)
torch.set_float32_matmul_precision(config.lightning.trainer.matmul_precision)

# Verify context arithmetic BEFORE anything is built
ctx_ch = config.velocity_net.params.context_in_channels * config.velocity_net.params.num_context_frames
print(f"context_in_channels : {config.velocity_net.params.context_in_channels}")
print(f"num_context_frames  : {config.velocity_net.params.num_context_frames}")
print(f"ctx_ch              : {ctx_ch}   ← must be 12")
assert ctx_ch == 12, f"ctx_ch={ctx_ch}, expected 12"
print("✓ Config OK")

Seed set to 2837


context_in_channels : 4
num_context_frames  : 3
ctx_ch              : 12   ← must be 12
✓ Config OK


In [12]:
pl.seed_everything(config.random.seed)
torch.set_float32_matmul_precision(
    config.lightning.trainer.get('matmul_precision', 'high')
)

vae_cfg           = LatentVAEConfig.from_config(config.vae)           
velocity_net_cfg  = VelocityNetConfig.from_config(config.velocity_net)
flow_matching_cfg = FlowMatchingConfig.from_config(config.flow_matching, config.sampling)
optimizer_cfg     = OptimizerConfig.from_config(config.optimizer, config.scheduler, config.train)

Seed set to 2837


# Data Module

In [13]:
train_cfg = config.train

base_data_module = WellDataModule(
    well_base_path     = well_base_path,
    well_dataset_name  = well_dataset_name,
    n_steps_input      = train_cfg.n_steps_input,
    n_steps_output     = train_cfg.n_steps_output,
    max_rollout_steps  = train_cfg.max_rollout_steps,
    batch_size         = train_cfg.batch_size,
    use_normalization  = False,
    normalization_type = None,
    transform          = compose_augmentation(*train_cfg.augmentations),
    data_workers       = 0,
    world_size         = 1,   # Lightning manages distribution
    rank               = 0,
)

data_module = WellDataModuleLightning(
    base_data_module,
    train_shuffle   = train_cfg.train_shuffle,
    val_shuffle     = train_cfg.val_shuffle,
    test_shuffle    = train_cfg.test_shuffle,
    rollout_shuffle = train_cfg.rollout_shuffle,
    target_dtype    = eval(train_cfg.target_dtype),
)

In [14]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

normalizer = ZScoreNormalization(
    stats,
    core_field_names          = base_data_module.train_dataset.core_field_names,
    core_constant_field_names = base_data_module.train_dataset.core_constant_field_names,
)

data_shape  = base_data_module.train_dataset.metadata.spatial_resolution
meta_fields = base_data_module.train_dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]
in_channels = len(field_names)


In [15]:
model = LatentFlowModule(                                                  # ← CHANGED (3)
    in_channels       = in_channels,
    field_names       = field_names,
    data_shape        = data_shape,
    n_steps_output    = train_cfg.n_steps_output,
    scale_weights     = config.loss.scale_weights,
    vae_cfg           = vae_cfg,                                           # ← CHANGED (3)
    velocity_net_cfg  = velocity_net_cfg,
    flow_matching_cfg = flow_matching_cfg,
    optimizer_cfg     = optimizer_cfg,
    normalizer        = normalizer,
)

# ── Experiment manager ───────────────────────────────────────────────────
exp_manager = None

#ExperimentManager(
#    config      = config,
#    model       = model,
#    field_names = field_names,
#)

model.experiment_manager = exp_manager

OSError: We couldn't connect to 'https://huggingface.co' to load this model, couldn't find it in the cached files and it looks like 'adamo1139/stable-diffusion-3.5-large-turbo-ungated' is not the path to a directory containing a config.json file.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/diffusers/installation#offline-mode'.

In [ ]:
  # ── Callbacks ────────────────────────────────────────────────────────────
callbacks = None #exp_manager.build_callbacks(config.callbacks.callbacks)

# ── Loggers ──────────────────────────────────────────────────────────────
loggers = None #exp_manager.build_loggers(config.loggers.loggers)

In [ ]:
lt_train = config.lightning.trainer
torch.set_float32_matmul_precision(lt_train.matmul_precision)
gradient_clip_val = lt_train.gradient_clip_val
gradient_clip_val = gradient_clip_val if isinstance(gradient_clip_val, float) else None

trainer = pl.Trainer(
    accelerator          = 'mps',
    strategy             = 'auto',
    devices              = 'auto',
    num_nodes            = 1,
    precision            = lt_train.precision,
    callbacks            = callbacks,
    max_epochs           = train_cfg.epochs,
    logger               = False,
    gradient_clip_val    = gradient_clip_val,
    log_every_n_steps    = lt_train.log_every_n_steps,
    num_sanity_val_steps = 0,
    limit_train_batches  = lt_train.limit_train_batches,
    limit_val_batches    = lt_train.limit_val_batches,
)


In [ ]:
trainer.fit(model, datamodule=data_module)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Typed configs
# ═══════════════════════════════════════════════════════════════
wavelet_cfg       = WaveletConfig.from_config(config.wavelet)
velocity_net_cfg  = VelocityNetConfig.from_config(config.velocity_net)
flow_matching_cfg = FlowMatchingConfig.from_config(config.flow_matching, config.sampling)
optimizer_cfg     = OptimizerConfig.from_config(config.optimizer, config.scheduler, config.train)

assert velocity_net_cfg.context_in_channels == 4, f"Got {velocity_net_cfg.context_in_channels}"
assert velocity_net_cfg.num_context_frames  == 3, f"Got {velocity_net_cfg.num_context_frames}"
print(f"velocity_net_cfg OK: context_in_channels={velocity_net_cfg.context_in_channels}, num_context_frames={velocity_net_cfg.num_context_frames}")

In [ ]:
velocity_net_cfg

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Data (num_workers=0 to fix macOS pickle error)
# ═══════════════════════════════════════════════════════════════
train_cfg = config.train

base_data_module = WellDataModule(
    well_base_path     = well_base_path,
    well_dataset_name  = well_dataset_name,
    n_steps_input      = train_cfg.n_steps_input,
    n_steps_output     = train_cfg.n_steps_output,
    max_rollout_steps  = train_cfg.max_rollout_steps,
    batch_size         = train_cfg.batch_size,
    use_normalization  = False,
    normalization_type = None,
    transform          = compose_augmentation(*train_cfg.augmentations),
    data_workers       = 10,
    world_size         = 1,
    rank               = 0,
)

data_module = WellDataModuleLightning(
    base_data_module,
    train_shuffle   = train_cfg.train_shuffle,
    val_shuffle     = train_cfg.val_shuffle,
    test_shuffle    = train_cfg.test_shuffle,
    rollout_shuffle = train_cfg.rollout_shuffle,
    target_dtype    = eval(train_cfg.target_dtype),
)

# Patch all dataloaders to force num_workers=0 (closures can't be pickled on macOS)
def _zero_workers(dl):
    if isinstance(dl, (list, tuple)):
        return [_zero_workers(d) for d in dl]
    return DataLoader(
        dataset     = dl.dataset,
        batch_size  = dl.batch_size,
        shuffle     = isinstance(dl.sampler, torch.utils.data.RandomSampler),
        num_workers = 0,
        collate_fn  = dl.collate_fn,
        pin_memory  = False,
        drop_last   = dl.drop_last,
    )

for method_name in ("train_dataloader", "val_dataloader", "test_dataloader", "predict_dataloader"):
    original = getattr(data_module, method_name, None)
    if original:
        setattr(data_module, method_name, lambda orig=original: _zero_workers(orig()))

data_module.setup('fit')

with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

normalizer = ZScoreNormalization(
    stats,
    core_field_names          = base_data_module.train_dataset.core_field_names,
    core_constant_field_names = base_data_module.train_dataset.core_constant_field_names,
)

data_shape  = base_data_module.train_dataset.metadata.spatial_resolution
meta_fields = base_data_module.train_dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]

print(f"Fields: {field_names} | Shape: {data_shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Model + verify stem channels BEFORE training
# ═══════════════════════════════════════════════════════════════
model = WaveletFlowModule(
    in_channels       = len(field_names),
    field_names       = field_names,
    data_shape        = data_shape,
    n_steps_output    = train_cfg.n_steps_output,
    scale_weights     = config.loss.scale_weights,
    wavelet_cfg       = wavelet_cfg,
    velocity_net_cfg  = velocity_net_cfg,
    flow_matching_cfg = flow_matching_cfg,
    optimizer_cfg     = optimizer_cfg,
    normalizer        = normalizer,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — MPS patch for adaptive_avg_pool2d
# ═══════════════════════════════════════════════════════════════
"""def _pool_context_cpu_fallback(self, ctx, H, W):
    B, C, T, Hf, Wf = ctx.shape
    x = ctx.permute(0, 2, 1, 3, 4).reshape(B * T, C, Hf, Wf)
    x = F.adaptive_avg_pool2d(x.cpu(), (H, W)).to(ctx.device)
    return x.reshape(B, C * T, H, W)

WaveletFlowUNet._pool_context = _pool_context_cpu_fallback
print("✓ MPS pool_context patch applied")"""

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Experiment manager + Trainer (local-safe version)
# ═══════════════════════════════════════════════════════════════

# Option A: just wipe the old experiment so no stale ckpt can be found
exp_dir = os.path.join(experiments_path, config.main.exp_name)
if os.path.exists(exp_dir):
    shutil.rmtree(exp_dir)
    print(f"Deleted stale experiment dir: {exp_dir}")

model.experiment_manager = ExperimentManager(
    base_dir           = experiments_path,
    experiment_name    = config.main.exp_name,
    overwrite          = True,   # <-- changed from False
    loggers            = config.loggers.loggers,
    #callbacks         = [],     # <-- strip ALL callbacks for local testing
    callbacks          = config.callbacks.callbacks,    
    log_every_n_epochs = config.loggers.log_every_n_epochs,
)

lt_train          = config.lightning.trainer
gradient_clip_val = lt_train.gradient_clip_val
gradient_clip_val = gradient_clip_val if isinstance(gradient_clip_val, float) else None

trainer = pl.Trainer(
    accelerator          = "mps",
    strategy             = "auto",
    devices              = 1,
    num_nodes            = 1,
    precision            = 'bf16-mixed',#"32",
    #callbacks            = [],   # no checkpoint/emergency/performance callbacks
    callbacks            = model.experiment_manager.get_callbacks(),
    max_epochs           = train_cfg.epochs,
    logger               = model.experiment_manager.get_loggers(),
    gradient_clip_val    = gradient_clip_val,
    log_every_n_steps    = 1,
    num_sanity_val_steps = 0,
    limit_train_batches  = 5,
    limit_val_batches    = 2,
)
print("✓ Trainer ready (no callbacks, no stale checkpoint)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Train
# ═══════════════════════════════════════════════════════════════
start_time = datetime.datetime.now()
print(f"Training started at {start_time}")

trainer.fit(model, datamodule=data_module)

end_time = datetime.datetime.now()
print(f"Done! Total time: {end_time - start_time}")